# *Dirty Dems* — Corpus EDA

An exploration of whether @democrats got meaner on TikTok.

The three accounts analyzed as part of this story are `@democrats` (the DNC), `@whitehouse` (the
Trump administration) and `@republicans` (the RNC). Every field comes from the two-stage LLM pipeline
(signals2text). Column definitions live in `data/DATA_DICTIONARY.md`.

## Setup

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 40)

# --- portable data root: walk up until we find data/posts ---
ROOT = Path.cwd()
for cand in (ROOT, *ROOT.parents):
    if (cand / "data" / "posts" / "democrats_posts.csv").exists():
        ROOT = cand; break
POSTS, CLS, AN = ROOT/"data"/"posts", ROOT/"data"/"classification", ROOT/"data"/"analysis"

ACCTS = ["democrats", "republicans", "whitehouse"]
NAME  = {"democrats": "@democrats", "republicans": "@republicans", "whitehouse": "@whitehouse"}
WINDOW = ["202602","202603","202604","202605","202606","202607"]   # months all three post
ELECTION = pd.Timestamp("2024-11-06")   # day after the 2024 election night

# posts metadata (one row per post)
posts = pd.concat([pd.read_csv(POSTS/f"{a}_posts.csv", dtype=str).assign(account=a)
                   for a in ACCTS], ignore_index=True)
posts["date"] = pd.to_datetime(posts["upload_date"], format="%Y%m%d")
posts["ym"]   = posts["upload_date"].str[:6]
for col in ["view_count","like_count","comment_count","repost_count"]:
    posts[col] = pd.to_numeric(posts[col], errors="coerce")

# classifier labels (one row per post)
labels = pd.concat([pd.read_json(CLS/f"function_{a}.jsonl", lines=True).assign(account=a)
                    for a in ACCTS], ignore_index=True)
labels["id"] = labels["video_id"].astype(str)

# one master table: metadata + labels
df = posts.merge(labels.drop(columns="account"), on="id", how="inner")

# a few tidy indicator columns, so later code is just groupby().mean()
df["crudeness"] = pd.to_numeric(df["crudeness"], errors="coerce")
df["era"]        = np.where(df["date"] < ELECTION, "pre-loss", "post-loss")
df["attack"]     = df["function"].eq("attack")
df["promote"]    = df["function"].isin(["promote_leader","promote_policy"])
df["crude"]      = df["crudeness"] >= 2
df["dunk"]       = df["has_dunk"].fillna(False).astype(bool)
df["rage_bait"]  = df["attention_mechanisms"].apply(
                       lambda m: "rage_bait_framing" in m if isinstance(m, list) else False)

print(f"{len(df)} classified posts loaded across {df['account'].nunique()} accounts")


4014 classified posts loaded across 3 accounts


One post can go after several people at once, so alongside the main table I build a
**treatments** table — one row for each person a post appraises. Anything about *who* a post targets
and *how warmly* (its `register`, and whose `side` they're on) comes from here.

In [2]:

# explode treatments -> one row per (post, person)
tr = df[["account","id","ym","era","treatments"]].explode("treatments")
tr = tr[tr["treatments"].notna()].reset_index(drop=True)
treat = pd.concat([tr[["account","id","ym","era"]],
                   pd.json_normalize(tr["treatments"])], axis=1)
print(f"{len(treat)} treatments (person-appraisals) across {treat['id'].nunique()} posts")
treat.head(3)


8812 treatments (person-appraisals) across 4014 posts


,account,id,ym,era,subject,side,register,is_main_character,is_peak,evidence
0,democrats,7668725528009903374,202607,post-loss,Donald Trump,opponent,3,True,True,Juxtaposed with an image of the devil and call...
1,democrats,7668429549662784781,202607,post-loss,Kash Patel,opponent,3,True,True,On-screen text highlights insults against him ...
2,democrats,7668412991318134030,202607,post-loss,U.S. State Department,own,2,True,True,A fabricated slide and overlay text mock the d...


## §1 — What's in the corpus

How many posts per account, over what stretch, and how uneven the three are. Dates are TikTok's
upload date (UTC, about five hours ahead of the East Coast), so a late-night post can land on the
next day — immaterial at a monthly grain. The accounts aren't the same size: the RNC only joined
TikTok in February 2026, so it's a sliver of the others. That's why every account-to-account
comparison (§5) stays inside February–July 2026, the only window all three post.

In [3]:

scope = df.groupby("account").agg(
    posts=("id", "size"),
    first=("date", "min"),
    last =("date", "max"),
).loc[ACCTS]
scope


,posts,first,last
account,,,
democrats,2881,2022-03-04,2026-07-31
republicans,225,2026-02-06,2026-07-31
whitehouse,908,2025-08-19,2026-07-31


In [4]:

n = df.groupby("account").size()
print(f"TOTAL corpus: {len(df)} posts")
print(f"Class imbalance (Dem : WH : RNC) = "
      f"{n['democrats']//n['republicans']}x : {n['whitehouse']//n['republicans']}x : 1x")


TOTAL corpus: 4014 posts
Class imbalance (Dem : WH : RNC) = 12x : 4x : 1x


## §2 — What each field looks like

The raw shape of the classifier's output, one field at a time, pooled across all three accounts and
all dates. Nothing's being compared yet — that comes in §5 and §6. One note: `register` is scored per
person, not per post, so its counts are treatments, not posts.

In [5]:

# FUNCTION — the primary purpose of a post (all values, not just 'attack')
(df["function"].value_counts(normalize=True).mul(100).round(1)
   .rename("share_%").to_frame())


,share_%
function,
attack,50.9
promote_leader,28.6
promote_policy,7.3
mobilize,7.1
ceremonial,2.9
patriotism,2.2
other,1.0


In [6]:

# CRUDENESS 0 (institutional) … 3 (crass)
(df["crudeness"].value_counts(normalize=True).mul(100).round(1)
   .sort_index().rename("share_%").to_frame())


,share_%
crudeness,
0,37.4
1,52.0
2,8.3
3,2.2


In [7]:

# HAS_DUNK — does the post land a quotable put-down?
print("dunk base rate:", f"{df['dunk'].mean()*100:.1f}%")

# REGISTER over all treatments (-3 worship … +3 hostility; +2 = attack line)
(treat["register"].value_counts(normalize=True).mul(100).round(1)
   .sort_index().rename("share_%").to_frame())


dunk base rate: 32.9%


,share_%
register,
-3,5.8
-2,28.4
-1,20.5
0,5.7
1,8.2
2,16.5
3,15.1


In [8]:

# ATTENTION MECHANISMS — share of posts using each (multi-label, so explode)
(df.explode("attention_mechanisms")["attention_mechanisms"].value_counts()
   .div(len(df)).mul(100).round(1).rename("share_of_posts_%").to_frame())


,share_of_posts_%
attention_mechanisms,
pop_culture_borrowing,48.7
meme_format,38.9
rage_bait_framing,33.9
static_text_card,22.4
aura_farming,18.9
screenshot_receipts,18.6
news_jacking,17.7
distortion_mockery,17.4
manipulation_fabrication,7.9


## §3 — When they post

Volume and rhythm, before any before/after cut. The staggered start dates — the DNC since 2022, the
White House since August 2025, the RNC since February 2026 — are just a fact of the data. For the
time of day I pull the timestamp baked into each TikTok post ID (it's the Unix time, independent of
the upload date); Eastern runs about four hours behind UTC in summer. A 9-to-5 rhythm here is the
sign of a staffed feed, not something the classifier decided.

In [9]:

# monthly post volume per account (wide table; last 14 months)
vol = (df.groupby(["ym","account"]).size().unstack("account")
         .reindex(columns=ACCTS))
vol.tail(14)


account,democrats,republicans,whitehouse
ym,,,
202506,42.0,NaN,NaN
202507,97.0,NaN,NaN
202508,87.0,NaN,48.0
202509,65.0,NaN,110.0
202510,95.0,NaN,82.0
202511,124.0,NaN,62.0
202512,75.0,NaN,63.0
202601,94.0,NaN,71.0
202602,119.0,32.0,71.0


In [10]:

# hour of day (UTC) from the post-ID timestamp
df["hour_utc"] = pd.to_datetime((df["id"].astype("int64") // 2**32), unit="s").dt.hour
by_hour = df["hour_utc"].value_counts(normalize=True).mul(100).round(1).sort_index()
et = (df["hour_utc"] - 4) % 24
print(f"Share posted 09:00–18:00 ET: {et.between(9, 18).mean()*100:.0f}%")
by_hour.rename("share_%").to_frame()


Share posted 09:00–18:00 ET: 81%


,share_%
hour_utc,
0,4.4
1,3.9
2,3.3
3,1.6
4,1.1
5,0.5
6,0.1
7,0.0
8,0.0


## §4 — How the fields move together

A few cross-tabs to see what tracks with what — still pooled across the accounts, so if something
shows up here it might be one account driving it. §5 checks that.

In [11]:

# mean crudeness by function
df.groupby("function")["crudeness"].mean().sort_values(ascending=False).round(2).to_frame("mean_crudeness")


,mean_crudeness
function,
attack,1.05
other,0.55
promote_policy,0.51
mobilize,0.47
promote_leader,0.45
patriotism,0.38
ceremonial,0.17


In [12]:

# dunk rate by crudeness level
(df.groupby("crudeness")["dunk"].mean().mul(100).round(1)).rename("dunk_rate_%").to_frame()


,dunk_rate_%
crudeness,
0,5.2
1,47.1
2,62.1
3,56.7


In [13]:

# mean register by side (own vs opponent vs neutral)
treat.groupby("side")["register"].agg(["mean","count"]).round(2).sort_values("mean", ascending=False)


,mean,count
side,,
opponent,2.09,3365
neutral,-0.72,1629
own,-1.73,3818


## §5 — The three accounts, side by side

The three compared inside February–July 2026, so they're measured over the same calendar time (the
fix for the lopsided sizing in §1). The RNC posts only 30–50 times a month here, so its rates are
rough.

In [14]:

window = df[df["ym"].isin(WINDOW)]
metrics = {"attack":"attack an opponent", "promote":"promote own side",
           "crude":"crude / crass", "dunk":"lands a dunk", "rage_bait":"rage-bait"}
rates = (window.groupby("account")[list(metrics)].mean().mul(100).round(0)
           .loc[ACCTS].rename(columns=metrics).T)
print("posts in window:", window.groupby("account").size().loc[ACCTS].to_dict())
rates


posts in window: {'democrats': 710, 'republicans': 225, 'whitehouse': 472}


account,democrats,republicans,whitehouse
attack an opponent,75.0,50.0,13.0
promote own side,17.0,34.0,68.0
crude / crass,19.0,18.0,8.0
lands a dunk,41.0,41.0,25.0
rage-bait,47.0,43.0,20.0


In [15]:

# average register aimed at opponents, in the window
(treat[(treat["side"] == "opponent") & treat["ym"].isin(WINDOW)]
   .groupby("account")["register"].mean().round(2).loc[ACCTS].rename("register_toward_opponents"))


account
democrats      2.25
republicans    2.28
whitehouse     2.10
Name: register_toward_opponents, dtype: float64

## §6 — The DNC before and after the loss

The DNC is the only account with a long pre-2024 record, so it's the only one I can cut before and
after the election. The line is November 6, 2024 — the day after election night. The pre-loss feed is
a campaign account with a candidate on it, so part of the shift is strategy and part is Harris leaving
the picture. "No argument" leans on a separate classifier that runs only on attacks — an attack with
nothing behind it, no accusation or policy point. It's @democrats-only.

In [16]:

dem = df[df["account"] == "democrats"].copy()
dem["mobilize"] = dem["function"].eq("mobilize")
cols = ["attack","promote","mobilize","crude","dunk"]
(dem.groupby("era")[cols].mean().mul(100).round(0)
   .loc[["pre-loss","post-loss"]].T)


era,pre-loss,post-loss
attack,40.0,78.0
promote,40.0,15.0
mobilize,17.0,3.0
crude,2.0,16.0
dunk,27.0,41.0


In [17]:

# no-argument share AMONG ATTACKS (charge classifier, DNC only)
charge = pd.read_json(CLS/"charge_democrats.jsonl", lines=True)
charge["id"] = charge["id"].astype(str)
dem = dem.merge(charge[["id","makes_charge"]], on="id", how="left")

attacks = dem[dem["attack"]].copy()
attacks["no_argument"] = ~attacks["makes_charge"].fillna(True).astype(bool)
(attacks.groupby("era")["no_argument"].mean().mul(100).round(0)
   .loc[["pre-loss","post-loss"]].rename("no_argument_%_of_attacks"))


era
pre-loss     21.0
post-loss    28.0
Name: no_argument_%_of_attacks, dtype: float64

**The exact number the story uses.** The headline stat — "about 9% → 20% → 25%" — is the same
no-argument attacks, but counted as a share of *all* posts in each period (not just among attacks),
split into three calendar chunks. Same posts, different denominator.

In [18]:

dem["pure_insult"] = dem["attack"] & ~dem["makes_charge"].fillna(True).astype(bool)
dem["period"] = np.select(
    [dem["date"] < ELECTION, dem["date"] < "2026-01-01"],
    ["pre-loss", "2025"], default="2026")
(dem.groupby("period")["pure_insult"].mean().mul(100).round(0)
   .loc["pre-loss 2025 2026".split()].rename("pure_insult_%_of_all_posts"))


period
pre-loss     9.0
2025        20.0
2026        25.0
Name: pure_insult_%_of_all_posts, dtype: float64

## §7 — Who's on screen

Just a frequency count of who each account features — no pre-set list, whoever shows up shows up.
Names appear exactly as the classifier wrote them, and I don't merge near-duplicates, so you can see
the raw split. For the DNC the denominator matters: Trump is in 44% of *all* its posts but 60% of its
*post-loss* posts, because the old campaign feed starred its own candidate. The story's "main
character … more than half" is about the post-loss feed, so I show both below.

In [19]:

featured = treat.drop_duplicates(["account","id","subject"])
n_posts  = df.groupby("account").size()

def top_cast(acct, k=8):
    counts = featured[featured["account"] == acct]["subject"].value_counts().head(k)
    return counts.div(n_posts[acct]).mul(100).round(1).rename("share_of_posts_%").to_frame()

top_cast("democrats")


,share_of_posts_%
subject,
Donald Trump,43.9
Kamala Harris,14.8
Joe Biden,13.1
Democratic Party,10.8
Republican Party,7.2
Barack Obama,5.0
Tim Walz,3.9
Jeffrey Epstein,3.0


In [20]:

# same for the two Republican accounts
top_cast("republicans")


,share_of_posts_%
subject,
Donald Trump,47.6
America,22.7
Democratic Party,19.6
Republican Party,8.0
James Talarico,7.6
the Democratic Party,5.3
Joe Biden,4.4
The Democratic Party,3.1


In [21]:

top_cast("whitehouse")


,share_of_posts_%
subject,
Donald Trump,68.9
America,27.9
Democratic Party,7.7
Melania Trump,7.5
The White House,6.2
Joe Biden,5.6
Donald J. Trump,5.6
Jd Vance,3.4


For the DNC I also collapse each person's name variants — the classifier writes "JD Vance,"
"Jd Vance" and "J.D. Vance" as three separate people — and label it as merged, so the numbers line up
with the ones the story names.

In [22]:

VARIANTS = {
    "Donald Trump":  "Donald Trump", "Donald J. Trump": "Donald Trump", "Trump": "Donald Trump",
    "JD Vance": "J.D. Vance", "Jd Vance": "J.D. Vance", "J.D. Vance": "J.D. Vance",
    "Kamala": "Kamala Harris", "Mamdani": "Zohran Mamdani", "Biden": "Joe Biden",
}
dem_feat = featured[featured["account"] == "democrats"].copy()
dem_feat["person"] = dem_feat["subject"].replace(VARIANTS)

names = ["Donald Trump","J.D. Vance","Kamala Harris","Barack Obama","Zohran Mamdani","Joe Biden"]
n_all  = (df["account"] == "democrats").sum()
n_post = ((df["account"] == "democrats") & (df["era"] == "post-loss")).sum()

def share(person, era=None):
    sub = dem_feat[dem_feat["person"] == person]
    if era: sub = sub[sub["era"] == era]
    denom = n_post if era == "post-loss" else n_all
    return round(sub["id"].nunique() / denom * 100, 1)

pd.DataFrame({
    "all_time_%":  [share(x) for x in names],
    "post_loss_%": [share(x, "post-loss") for x in names],
}, index=names)


,all_time_%,post_loss_%
Donald Trump,44.8,62.0
J.D. Vance,3.6,4.3
Kamala Harris,14.8,1.4
Barack Obama,5.0,4.6
Zohran Mamdani,1.9,3.3
Joe Biden,13.1,2.6


**The exact number the story uses.** Trump's share of the post-loss feed — the
"main character … more than half" line.

In [23]:

tr_dem = treat[treat["account"] == "democrats"].copy()
tr_dem["person"] = tr_dem["subject"].replace(VARIANTS)
post = tr_dem[tr_dem["era"] == "post-loss"]

trump = post[post["person"] == "Donald Trump"]
featured_pct = trump["id"].nunique() / n_post * 100
main_pct     = trump[trump["is_main_character"] == True]["id"].nunique() / n_post * 100
print(f"Trump, POST-LOSS @democrats (denominator = {n_post} posts):")
print(f"   featured (any treatment): {featured_pct:.1f}%")
print(f"   main character:           {main_pct:.1f}%   <- 'more than half'")


Trump, POST-LOSS @democrats (denominator = 1619 posts):
   featured (any treatment): 62.0%
   main character:           59.2%   <- 'more than half'


## §8 — Does the meanness travel?

For @democrats, I split the feed into attack posts and everything else, and take the median (the
middle value) of views, likes, comments and reposts for each. The bottom row divides one by the
other — how many times further an attack travels than an ordinary post. So an 11 in the repost column
means the DNC's attacks get reposted about eleven times as often as its other posts.

Medians, not averages, so one viral post doesn't skew it.

In [24]:

dem_eng = df[df["account"] == "democrats"]
cols = ["view_count", "like_count", "comment_count", "repost_count"]

med = dem_eng.groupby("attack")[cols].median()
ratio = (med.loc[True] / med.loc[False]).round(1)     # attack vs non-attack, before relabeling
med.index = ["non-attack posts", "attack posts"]      # attack=False, attack=True
med = med.round(0)
med.loc["ratio (attack ÷ non-attack)"] = ratio
med


,view_count,like_count,comment_count,repost_count
non-attack posts,110200.0,7609.0,111.0,124.0
attack posts,377500.0,33000.0,268.0,1343.0
ratio (attack ÷ non-attack),3.4,4.3,2.4,10.8


## §9 — Caution on some figures

1) the manipulation flag is unreliable
The model has no world knowledge — it calls real events fake, and now and then invents manipulation. 

2) the `track` metadata is polluted
Track metadata is not that reliable. Music identity should come from ACRCloud fingerprints, never this column. But anyway this is also not that central to the story.

3) @republicans is small
The RNC's TikTok feed has 225 posts total, ~30–50 a month. Its monthly rates wobble.


In [25]:

# manipulation flag
flagged = df["attention_mechanisms"].apply(
    lambda m: "manipulation_fabrication" in m if isinstance(m, list) else False)
print(f"manipulation flag: {flagged.sum()} posts flagged ({flagged.mean()*100:.1f}%)")

# `track` metadata
track = posts["track"].fillna("").str.lower()
generic = track.str.contains("original sound|sonido original|som original|оригинал", regex=True)
print(f"`track` metadata: {generic.sum()} of {(track != '').sum()} non-empty values are generic")

manipulation flag: 318 posts flagged (7.9%)
`track` metadata: 3159 of 3969 non-empty values are generic


4) Profanity is low across all three accounts (authored rate, %)

In [26]:

print(pd.read_csv(AN/"profanity_by_period.csv")[["account","period","posts","authored_rate_%"]]
        .to_string(index=False))

    account   period  posts  authored_rate_%
  democrats pre-loss   1262              1.2
  democrats     2025    815              5.5
  democrats     2026    804              6.8
 whitehouse     2025    365              3.3
 whitehouse     2026    543              4.1
republicans     2026    225              6.2
